# Bidirectional RNN - TensorFlow / Keras


In [1]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Generate synthetic dataset
samples, seq_len, features = 1400, 30, 3
X = np.random.normal(size=(samples, seq_len, features)).astype("float32")

signal = (X[:, -10:, 0].mean(axis=1) + 0.5 * X[:, :10, 1].mean(axis=1))
y = (signal > 0.05).astype("int64")

# Train/test split
X_train, X_test = X[:1000], X[1000:]
y_train, y_test = y[:1000], y[1000:]

# Define model
model = keras.Sequential([
    layers.Input(shape=(seq_len, features)),
    layers.Bidirectional(layers.GRU(32)),
    layers.Dropout(0.2),
    layers.Dense(2, activation="softmax")])

# Compile model
model.compile(
    optimizer=keras.optimizers.AdamW(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Model summary
model.summary()

# Train model
model.fit(X_train, y_train,
          validation_data=(X_test, y_test),
          epochs=10,
          batch_size=64
          )

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 64)             │         7,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,234 (28.26 KB)

 Trainable params: 7,234 (28.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.5160 - loss: 0.7000 - val_accuracy: 0.6300 - val_loss: 0.6542
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6500 - loss: 0.6412 - val_accuracy: 0.7150 - val_loss: 0.6074
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7410 - loss: 0.5885 - val_accuracy: 0.7625 - val_loss: 0.5448
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7990 - loss: 0.4983 - val_accuracy: 0.8350 - val_loss: 0.4405
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8390 - loss: 0.4080 - val_accuracy: 0.8450 - val_loss: 0.3732
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8620 - loss: 0.3522 - val_accuracy: 0.8775 - val_loss: 0.3170
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8800 - loss: 0.2984 - val_accuracy: 0.8675 - val_loss: 0.2876
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9050 - loss: 0.2694 - val_accuracy: 0.8825 - v

In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding,
                                     SimpleRNN,
                                     Dense,
                                     Bidirectional,
                                     Dropout
                                     )

# -----------------------------------
# 1. Hyperparameters
# -----------------------------------
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBED_DIM = 128
RNN_UNITS = 64
BATCH_SIZE = 64
EPOCHS = 5

# -----------------------------------
# 2. Load Dataset
# -----------------------------------
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# -----------------------------------
# 3. Padding Sequences
# -----------------------------------
x_train = pad_sequences(
    x_train,
    maxlen=MAX_LEN,
    padding='post'
)

x_test = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding='post'
)

# -----------------------------------
# 4. Build Bidirectional RNN Model
# -----------------------------------
model = Sequential([

    # Embedding Layer
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),

    # Bidirectional RNN
    Bidirectional(
        SimpleRNN(
            RNN_UNITS,
            return_sequences=False
        )
    ),

    Dropout(0.5),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

# -----------------------------------
# 5. Compile Model
# -----------------------------------
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -----------------------------------
# 6. Model Summary
# -----------------------------------
model.summary()

# -----------------------------------
# 7. Train Model
# -----------------------------------
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

# -----------------------------------
# 8. Evaluate Model
# -----------------------------------
test_loss, test_acc = model.evaluate(
    x_test,
    y_test
)

print(f"\nTest Accuracy: {test_acc:.4f}")

# -----------------------------------
# 9. Prediction Example
# -----------------------------------
sample = x_test[0:1]

prediction = model.predict(sample)

if prediction[0][0] > 0.5:
    print("Positive Review")
else:
    print("Negative Review")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.5077 - loss: 0.7014 - val_accuracy: 0.4958 - val_loss: 0.7190
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.5739 - loss: 0.6761 - val_accuracy: 0.5034 - val_loss: 0.7364
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.7014 - loss: 0.5892 - val_accuracy: 0.7536 - val_loss: 0.5335
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.6339 - loss: 0.6351 - val_accuracy: 0.5448 - val_loss: 0.6914
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.5720 - loss: 0.6782 - val_accuracy: 0.5602 - val_loss: 0.6849
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.5498 - loss: 0.6860

Test Accuracy: 0.5498
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 523ms/step
Negative Review
